# 🤖 Forex Trading Bot — ML Based
### Running di Google Colab

Notebook ini menjalankan seluruh stack:
- **Redis** — message broker untuk Celery
- **Celery Worker** — background task processor
- **FastAPI** — REST API (port 8000)
- **Flask Web Dashboard** — UI dashboard (port 5000)
- **ngrok** — expose port supaya bisa diakses dari browser

> Jalankan setiap cell secara berurutan dari atas ke bawah.

---
## 1. Clone Repository

In [ ]:
import os

REPO_DIR = "/content/forex_trading_bot_ml_based"

if os.path.exists(REPO_DIR):
    print("Repo sudah ada, pull latest changes...")
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Herutriana44/forex_trading_bot_ml_based.git {REPO_DIR}
    print("Clone selesai!")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

---
## 2. Install Dependencies

In [ ]:
# Install semua Python dependencies dari requirements.txt
!pip install -q -r requirements.txt

# Install Flask + SocketIO untuk web dashboard
!pip install -q flask flask-socketio

# Install pyngrok untuk expose port ke internet
!pip install -q pyngrok

print("✅ Semua dependencies berhasil diinstall.")

---
## 3. Install & Jalankan Redis

In [ ]:
# Install Redis di Colab (Ubuntu-based)
!apt-get install -qq redis-server > /dev/null 2>&1

# Jalankan Redis di background
!redis-server --daemonize yes

import time
time.sleep(2)

# Verifikasi Redis berjalan
result = !redis-cli ping
if 'PONG' in result[0]:
    print("✅ Redis berjalan dengan baik!")
else:
    print("❌ Redis gagal start:", result)

---
## 4. Setup Environment Variables

In [ ]:
import os

# Pastikan working directory benar
os.chdir("/content/forex_trading_bot_ml_based")

# Set environment variables
os.environ["REDIS_URL"] = "redis://localhost:6379/0"
os.environ["CELERY_RESULT_BACKEND"] = "redis://localhost:6379/1"
os.environ["DATABASE_URL"] = "sqlite:////content/forex_trading_bot_ml_based/src/db/trading.db"
os.environ["RETRAIN_ACCURACY_THRESHOLD"] = "0.55"
os.environ["PYTHONPATH"] = "/content/forex_trading_bot_ml_based"

# Buat folder models jika belum ada
os.makedirs("src/models/versioned", exist_ok=True)
os.makedirs("src/models/current", exist_ok=True)
os.makedirs("src/db", exist_ok=True)

print("✅ Environment variables berhasil di-set.")

---
## 5. Inisialisasi Database

In [ ]:
import sys
sys.path.insert(0, "/content/forex_trading_bot_ml_based")

# Import dan inisialisasi database (buat tabel jika belum ada)
from src.db.logging import Base, engine
Base.metadata.create_all(bind=engine)

print("✅ Database berhasil diinisialisasi.")

---
## 6. Training Model Pertama Kali

> Wajib dijalankan sebelum prediksi. Proses download data dari Yahoo Finance dan melatih model XGBoost.

In [ ]:
import sys
sys.path.insert(0, "/content/forex_trading_bot_ml_based")
os.chdir("/content/forex_trading_bot_ml_based")

from src.retraining.pipeline import run_retraining_pipeline

print("⏳ Memulai training model untuk EURUSD=X...")
print("   (Download data dari Yahoo Finance + feature engineering + training XGBoost)")
print()

results = run_retraining_pipeline(symbol="EURUSD=X", start_date="2019-01-01")

print()
print("✅ Training selesai!")
print(f"   Akurasi    : {results.get('accuracy', 'N/A')}")
print(f"   Precision  : {results.get('precision', 'N/A')}")
print(f"   Recall     : {results.get('recall', 'N/A')}")
print(f"   Versi model: {results.get('version', 'N/A')}")

---
## 7. Jalankan Celery Worker (Background)

In [ ]:
import subprocess, time

celery_proc = subprocess.Popen(
    [
        "python", "-m", "celery",
        "-A", "src.tasks.celery_app", "worker",
        "--loglevel=info",
        "--concurrency=2"
    ],
    cwd="/content/forex_trading_bot_ml_based",
    env={**os.environ, "PYTHONPATH": "/content/forex_trading_bot_ml_based"},
    stdout=open("/tmp/celery.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

if celery_proc.poll() is None:
    print(f"✅ Celery Worker berjalan. PID: {celery_proc.pid}")
else:
    print("❌ Celery Worker gagal start. Cek log di bawah:")
    !cat /tmp/celery.log

---
## 8. Jalankan FastAPI Server (Background)

In [ ]:
import subprocess, time

fastapi_proc = subprocess.Popen(
    [
        "python", "-m", "uvicorn",
        "src.api.main:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    cwd="/content/forex_trading_bot_ml_based",
    env={**os.environ, "PYTHONPATH": "/content/forex_trading_bot_ml_based"},
    stdout=open("/tmp/fastapi.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(4)

if fastapi_proc.poll() is None:
    print(f"✅ FastAPI Server berjalan di port 8000. PID: {fastapi_proc.pid}")
else:
    print("❌ FastAPI gagal start. Cek log di bawah:")
    !cat /tmp/fastapi.log

---
## 9. Jalankan Flask Web Dashboard (Background)

In [ ]:
import subprocess, time

flask_proc = subprocess.Popen(
    ["python", "-m", "src.web.app"],
    cwd="/content/forex_trading_bot_ml_based",
    env={**os.environ, "PYTHONPATH": "/content/forex_trading_bot_ml_based"},
    stdout=open("/tmp/flask.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(4)

if flask_proc.poll() is None:
    print(f"✅ Flask Dashboard berjalan di port 5000. PID: {flask_proc.pid}")
else:
    print("❌ Flask gagal start. Cek log di bawah:")
    !cat /tmp/flask.log

---
## 10. Expose ke Internet via ngrok

> Daftarkan akun gratis di [ngrok.com](https://ngrok.com) dan dapatkan authtoken di https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok

# ⚠️ Ganti dengan authtoken kamu dari https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Expose FastAPI (port 8000)
api_tunnel = ngrok.connect(8000)
print(f"🌐 FastAPI (REST API) : {api_tunnel.public_url}")
print(f"   Swagger Docs       : {api_tunnel.public_url}/docs")
print(f"   ReDoc              : {api_tunnel.public_url}/redoc")
print()

# Expose Flask Dashboard (port 5000)
web_tunnel = ngrok.connect(5000)
print(f"🖥️  Flask Dashboard    : {web_tunnel.public_url}")

---
## 11. Test API — Cek Status Model

In [ ]:
import requests, json

BASE_URL = "http://localhost:8000/api/v1"

# Cek status model
response = requests.get(f"{BASE_URL}/model/status")
print("📊 Model Status:")
print(json.dumps(response.json(), indent=2))

---
## 12. Test API — Buat Prediksi

In [ ]:
import requests, json, time

BASE_URL = "http://localhost:8000/api/v1"

# Kirim request prediksi
symbol = "EURUSD=X"  # Ganti sesuai kebutuhan: GBPUSD=X, USDJPY=X, dll.

print(f"⏳ Membuat prediksi untuk {symbol}...")
resp = requests.post(f"{BASE_URL}/predict", json={"symbol": symbol})
task_info = resp.json()
task_id = task_info["task_id"]
print(f"   Task ID: {task_id}")

# Polling sampai hasil tersedia
for attempt in range(30):
    time.sleep(3)
    result = requests.get(f"{BASE_URL}/predict/{task_id}").json()
    status = result.get("status")
    print(f"   [{attempt+1}] Status: {status}")

    if status == "success":
        print()
        print("✅ Hasil Prediksi:")
        print(json.dumps(result, indent=2))
        break
    elif status == "error":
        print("❌ Error:", result.get("error"))
        break
else:
    print("⚠️ Timeout menunggu hasil prediksi.")

---
## 13. Test API — Trigger Retraining Model

In [ ]:
import requests, json, time

BASE_URL = "http://localhost:8000/api/v1"

symbol = "EURUSD=X"
start_date = "2019-01-01"

print(f"⏳ Memulai retraining untuk {symbol} dari {start_date}...")
resp = requests.post(
    f"{BASE_URL}/model/retrain",
    json={"symbol": symbol, "start_date": start_date}
)
task_info = resp.json()
task_id = task_info["task_id"]
print(f"   Task ID: {task_id}")

# Polling sampai selesai
for attempt in range(40):
    time.sleep(5)
    result = requests.get(f"{BASE_URL}/model/retrain/{task_id}").json()
    status = result.get("status")
    print(f"   [{attempt+1}] Status: {status}")

    if status == "success":
        print()
        print("✅ Retraining selesai!")
        print(json.dumps(result, indent=2))
        break
    elif status == "error":
        print("❌ Error:", result.get("error"))
        break
else:
    print("⚠️ Timeout menunggu retraining selesai.")

---
## 14. Cek Log Semua Service

In [ ]:
print("===== LOG FASTAPI (50 baris terakhir) =====")
!tail -50 /tmp/fastapi.log

print("\n===== LOG CELERY (50 baris terakhir) =====")
!tail -50 /tmp/celery.log

print("\n===== LOG FLASK (50 baris terakhir) =====")
!tail -50 /tmp/flask.log

---
## 15. Stop Semua Service

> Jalankan cell ini jika ingin mematikan semua proses.

In [ ]:
from pyngrok import ngrok as _ngrok

# Tutup semua tunnel ngrok
try:
    _ngrok.kill()
    print("✅ Ngrok tunnels ditutup.")
except Exception as e:
    print(f"⚠️ Ngrok: {e}")

# Hentikan proses
for name, proc in [("Flask", flask_proc), ("FastAPI", fastapi_proc), ("Celery", celery_proc)]:
    try:
        proc.terminate()
        proc.wait(timeout=5)
        print(f"✅ {name} dihentikan.")
    except Exception as e:
        print(f"⚠️ {name}: {e}")

# Stop Redis
!redis-cli shutdown nosave 2>/dev/null || true
print("✅ Redis dihentikan.")